# Adult-Onset Hereditary Ataxia Variant Analysis
**Author:** [Isaac Li]  
**Target Publication:** Journal of Emerging Investigators (JEI)  
**Data Sources:** NCBI ClinVar (GRCh38), Genomics England PanelApp (Panel 466)  

### Overview
This notebook reproduces the data retrieval, filtering, HGVS consequence classification, and statistical analysis comparing predicted loss-of-function (LoF) vs. missense variants across 174 adult-onset hereditary ataxia genes.

In [ ]:
import pandas as pd
import urllib.request
import os
import math
from scipy.stats import chi2_contingency

In [ ]:
# Complete list of the 174 Green-rated genes from
# Genomics England PanelApp Adult-Onset Hereditary Ataxia panel (Panel 466)

ataxia_genes = [ 'AAAS', 'ABHD12', 'ADCY5', 'AFG3L2', 'ANO10', 'APTX', 'ARMC9', 'ARSA', 'ATCAY', 'ATM',
    'ATN1', 'ATP1A2', 'ATP1A3', 'ATP7B', 'ATXN1', 'ATXN10', 'ATXN2', 'ATXN3', 'ATXN7', 'AUH',
    'B4GALNT1', 'C12orf65', 'CACNA1A', 'CACNA1G', 'CAPN1', 'CA8', 'CWF19L1', 'DAG1', 'DARS2',
    'DNMT1', 'ELOVL4', 'ELOVL5', 'EIF2B1', 'EIF2B2', 'EIF2B3', 'EIF2B4', 'EIF2B5', 'FA2H',
    'FGF14', 'FLVCR1', 'FMR1', 'FOXRED1', 'FXN', 'GBA1', 'GDAP1', 'GJB1', 'GRID2', 'GRM1',
    'INPP5E', 'ITPR1', 'KCNA1', 'KCNC3', 'KCNJ10', 'KCNMA1', 'KCNQ2', 'KMT2B', 'LGI1',
    'MARS2', 'MTPAP', 'NOP56', 'NOTCH3', 'NPC1', 'NPC2', 'NDUFV1', 'OPTN', 'PANK2', 'PDXA',
    'PEX10', 'PEX7', 'POLG', 'PPP2R2B', 'PRKCG', 'PRNP', 'PRRT2', 'PMP22', 'PNKP', 'PNPLA6',
    'POLR3A', 'POLR3B', 'PPT1', 'PURA', 'REEP1', 'RNASEH2A', 'RNASEH2B', 'RNASEH2C', 'SAMHD1',
    'SACS', 'SETX', 'SIL1', 'SLC1A3', 'SLC2A1', 'SLC9A6', 'SNX14', 'SPG7', 'SPTBN2', 'STUB1',
    'SYNE1', 'TBP', 'TDP1', 'TGM6', 'TMEM240', 'TPP1', 'TREX1', 'TTBK2', 'TTPA', 'TUBB4A',
    'UBTF', 'VCP', 'VPS13D', 'WFS1', 'ZFYVE26'
]

print(f"Number of PanelApp genes loaded: {len(ataxia_genes)}")

url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
filename = "variant_summary.txt.gz"

print("Downloading ClinVar Variant Summary...")

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

print("Loading ClinVar data...")

use_cols = [
    'VariationID',
    'Name',
    'GeneSymbol',
    'Type',
    'ClinicalSignificance',
    'Assembly'
]

df = pd.read_csv(
    filename,
    sep='\t',
    compression='gzip',
    usecols=use_cols,
    low_memory=False
)

print("Filtering for GRCh38 and selected PanelApp genes...")

df_filtered = df[
    (df['Assembly'] == 'GRCh38') &
    (df['GeneSymbol'].isin(ataxia_genes))
].copy()

print(f"Variants before deduplication: {len(df_filtered):,}")

In [ ]:
# Keep only one record for each unique ClinVar VariationID
df_filtered = df_filtered.drop_duplicates(
    subset='VariationID'
).copy()

print(f"Variants after deduplication: {len(df_filtered):,}")

In [ ]:
def classify_consequence_v4(row):
    name = str(row['Name']) # Keep original name
    name_lower = name.lower()

    # Loss-of-function candidates:
    if (
        'fs' in name_lower
        or 'ter' in name_lower
        or '*' in name
        or any(splice in name_lower for splice in ['+1', '+2', '-1', '-2'])
    ):
        return 'Loss-of-Function (LoF)'

    # Missense:
    # Looks for 'p.' in the original string and ensures it's not synonymous ('=')
    elif 'p.' in name and '=' not in name:
        return 'Missense'

    else:
        return 'Other'

In [ ]:
df_filtered['Refined_Consequence_v4'] = df_filtered.apply(
    classify_consequence_v4,
    axis=1
)

print(df_filtered['Refined_Consequence_v4'].value_counts())

In [ ]:
df_study_v4 = df_filtered[
    df_filtered['Refined_Consequence_v4'].isin(
        ['Missense', 'Loss-of-Function (LoF)']
    )
].copy()

print(f"Final LoF + missense dataset: {len(df_study_v4):,}")

In [ ]:
def classify_primary_pathogenicity(value):
    value = str(value).strip()

    if value in [
        'Pathogenic',
        'Likely pathogenic',
        'Pathogenic/Likely pathogenic'
    ]:
        return True
    else:
        return False

In [ ]:
df_study_v4['Primary_Pathogenic'] = (
    df_study_v4['ClinicalSignificance']
    .apply(classify_primary_pathogenicity)
)

In [ ]:
contingency_table = pd.crosstab(
    df_study_v4['Refined_Consequence_v4'],
    df_study_v4['Primary_Pathogenic']
)

contingency_table

In [ ]:
table = pd.crosstab(
    df_study_v4['Refined_Consequence_v4'],
    df_study_v4['Primary_Pathogenic']
)

print("df_study_v4 shape:", df_study_v4.shape)

print("\nConsequence counts:")
print(df_study_v4['Refined_Consequence_v4'].value_counts(dropna=False))

print("\nPrimary pathogenicity counts:")
print(df_study_v4['Primary_Pathogenic'].value_counts(dropna=False))

print("\nContingency table:")
print(table)

In [ ]:
table = pd.crosstab(
    df_study_v4['Refined_Consequence_v4'],
    df_study_v4['Primary_Pathogenic']
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)

print("\nExpected counts:")
print(expected)

In [ ]:
def classify_clinical_significance_final(value):
    value = str(value).strip()
    lower_value = value.lower()

    # Exact pathogenic definition used in the primary analysis
    if value in [
        'Pathogenic',
        'Likely pathogenic',
        'Pathogenic/Likely pathogenic'
    ]:
        return 'Pathogenic/Likely Pathogenic'

    elif 'conflicting' in lower_value:
        return 'Conflicting Interpretations'

    elif 'uncertain significance' in lower_value:
        return 'VUS'

    elif 'benign' in lower_value:
        return 'Benign/Likely Benign'

    else:
        return 'Other'

In [ ]:
df_study_v4['ClinicalCategory_final'] = (
    df_study_v4['ClinicalSignificance']
    .apply(classify_clinical_significance_final)
)

In [ ]:
secondary_table_final = pd.crosstab(
    df_study_v4['Refined_Consequence_v4'],
    df_study_v4['ClinicalCategory_final']
)

secondary_table_final

In [ ]:
secondary_percentages = secondary_table_final.loc[
    ['Loss-of-Function (LoF)', 'Missense']
].div(
    secondary_table_final.loc[
        ['Loss-of-Function (LoF)', 'Missense']
    ].sum(axis=1),
    axis=0
).mul(100).round(2)

secondary_percentages